# Modelado Predictivo de Demanda Diaria (Hospital Universidad del Norte)

Este notebook implementa y evalúa la **escalera de modelos de demanda diaria** por
tipo de visita / servicio para el Hospital Universidad del Norte (HUN), con un horizonte
de predicción de **1 a 7 días**.

## Objetivo y Enfoque
- **Escalera de modelos:**
  1. *Línea base:* mediana histórica del mismo día de la semana en las 8 semanas anteriores.
  2. *Poisson con calendario:* regresión de Poisson que captura día de la semana, mes,
     estacionalidad intra-anual y eventos festivos.
  3. *LightGBM (todos los externos):* gradient boosting con pérdida Poisson, rezagos
     semanales anti-fuga (7 a 56 días) y la totalidad de variables externas.
  4. *LightGBM final:* retiene únicamente los grupos de factores externos aceptados mediante
     protocolo de ablación y estabilidad.
- **Validación walk-forward:** evaluación estrictamente temporal en pliegues cronológicos,
  sin mezclar pasado y futuro.
- **Anti-fuga estricta:** rezagos de al menos 7 días y enmascaramiento de observaciones
  posteriores al origen de pronóstico.
- **Análisis en festivos:** evaluación específica del comportamiento del modelo en días
  festivos colombianos frente a la línea base.

In [ ]:
import logging

import pandas as pd

from hun import config
from hun.data.agregados import agrupar_bajo_volumen, fecha_corte, serie_llegadas
from hun.models import demanda, validacion
from hun.models import evaluacion_factores as ef

logging.basicConfig(level=logging.INFO, format="%(message)s")

## 1. Carga y preparación de series de llegadas diarias

Cargamos la extracción agregada `entradas_por_hora.parquet`, aplicamos la fecha de corte
detectada automáticamente, agregamos las llegadas a nivel diario (`freq="D"`) y agrupamos
servicios con menos de 1 llegada diaria promedio.

In [ ]:
ruta_entradas = config.RAW_DIR / "entradas_por_hora.parquet"
ruta_externos = config.DATA_DIR / "externos.parquet"

if ruta_entradas.exists():
    entradas = pd.read_parquet(ruta_entradas)
    corte = fecha_corte(entradas)
    entradas = entradas[pd.to_datetime(entradas["hora"]) < corte]
    serie = agrupar_bajo_volumen(serie_llegadas(entradas, freq="D"), min_por_dia=1.0)
else:
    print("Nota: entradas_por_hora.parquet no encontrado en data/raw/.")
    serie = None

## 2. Factores externos y pliegues walk-forward

Cargamos la base de datos externa unificada `data/externos.parquet` (calendario, clima y
epidemiología con rezago anti-fuga) y construimos los pliegues temporales walk-forward.

In [ ]:
externos = None
if ruta_externos.exists():
    externos = pd.read_parquet(ruta_externos)
    if "fecha" in externos.columns:
        externos = externos.set_index("fecha")

if serie is not None:
    pliegues = validacion.pliegues_walk_forward(serie["ts"])
    print(f"Pliegues generados: {len(pliegues)}")

## 3. Protocolo de evaluación empírica de factores externos

Ejecutamos el protocolo de ablación con LightGBM (`ajustar_predecir_lgbm`), control de
estacionalidad, prueba placebo y estabilidad para verificar qué grupos de factores
aportan mejora estadística real.

In [ ]:
if serie is not None and externos is not None:
    resultados_grupos = ef.evaluar_grupos(
        serie,
        externos,
        pliegues,
        ajustar_predecir=demanda.ajustar_predecir_lgbm,
    )
    tabla_grupos = ef.tabla_resultados(resultados_grupos)
    print(tabla_grupos.to_string())

## 4. Evaluación de la escalera de modelos

Comparamos la línea base contra Poisson y LightGBM sobre los mismos pliegues temporales,
analizando el WAPE, MAE y sesgo de predicción.

In [ ]:
if serie is not None:
    resultados_escalera = demanda.evaluar_escalera(
        serie,
        externos,
        pliegues,
        grupos_aceptados=demanda.GRUPOS_ACEPTADOS,
    )
    reporte_md = demanda.reporte_markdown(resultados_escalera, pliegues=pliegues)
    print(reporte_md)

## 5. Desglose detallado del modelo final (festivos y servicios)

Analizamos el comportamiento de LightGBM Poisson final en días festivos vs. días
normales y por servicio individual.

In [ ]:
if serie is not None and "lgbm_final" in resultados_escalera:
    resumen = demanda.resumen_completo(resultados_escalera["lgbm_final"])
    print("--- Desempeño Global ---")
    print(resumen["global"].to_string())
    print("\n--- Desempeño en Festivos vs. Normales ---")
    print(resumen["por_festivo"].to_string())
    print("\n--- Desempeño por Servicio ---")
    print(resumen["por_serie"].to_string())